# Get all the Gartenlaube Text in Wikisource

Mehr als 15.000 Artikel der Gartenlaube sind Ende 2021 in Wikisource erschlossen, ein vielfaches an Seiten der Illustrierten stehen hinter diesen Artikeln. 

## Frage 

Wie kommt man zum Texttranskript aller Gartenlaube-Artikel in Wikisource? (vgl. https://de.wikisource.org/wiki/Benutzer_Diskussion:Jeb#Gartenlaube_runterladen)

## Let's hack

Die MediaWiki-API bietet eine Vielzahl an Methoden um Daten aus MediaWikis - wie Wikisource - maschinell und im großen Stil zu extrahieren. Für große Projekte wie die Gartenlaube, die letztlich auch so sturkturiert sind wie sie strukturiert sind, geht es aber nicht ganz ohne Code. 

Was muss vorab beachtet werden?

* Der Text von Wikisource-Artikeln liegt nicht in den Seiten der strukturierten Zeitschriftenartikel, sondern in den Wiki-Artikel des `Seite:`-Namespaces
* Ein Wikisource-Großprojekt wie Die Gartenlaube ist nicht in einer einzigen Projektkategorie organisiert. Die einzelnen Seiten liegen in Jahrgangskategorien vor, die selbst eine Unterkategorie der Gartenlaube-Kategorie sind.
* Die MediaWiki-API bietet eine Extension an, um möglichst "Plain"-Text zu erhalten. Diese Extension `TextExctracts`(https://www.mediawiki.org/wiki/Extension:TextExtracts/de) ist aber für die Wikisource nicht verfügbar, da hier die Extension Proofread dies technisch gegenwärtig nicht ermöglicht. Daher ist ein Text-Output nur in einem gerenderte HTML oder im Wikitext möglich. Beide Varianten werden am Ende dieses Skripts im Output vereint.
* Mit der Python-Library `mwparserfromhell` wird noch zusätzlich ein plaintext ausgegeben, dieser Text basiert auf dem Wikicode, verliert aber sämtliche Textinhalte, die in Vorlagen gespeichert waren. Konkret sind dies Überschriften und Bildunterschriften. Dieser Volltext ist daher nur als Referenzwert zu verstehen und mit Vorsicht zu genießen.

### 1. Schritt - Alle Jahrgangskategorien parsen

In [1]:
import requests

S = requests.Session()

URL = "https://de.wikisource.org/w/api.php"

PARAMS = {
    "action": "query",
    "cmtitle": "Kategorie:Die Gartenlaube",
    "cmlimit":500,
    "cmtype": "subcat",
    "list": "categorymembers",
    "format": "json"
}

R = S.get(url=URL, params=PARAMS)
DATA = R.json()

Gartenlaube_SubCat = DATA["query"]["categorymembers"]

#for SubCat in Gartenlaube_SubCat:
#    print(SubCat["title"])

### 2. Für jede Subkategorie der Gartenlaube, Seiten des "Seite:"-Namespaces abrufen

In [2]:

pages = []
retDict = {}
for SubCat in Gartenlaube_SubCat:
    catPages = []
    PARAMS = {
    "action": "query",
    "gcmtitle": SubCat["title"],
    "gcmlimit": 500,
    "gcmnamespace":102,
    "generator": "categorymembers",
    "prop": "proofread|info",
    "format": "json"
    }

    R = S.get(url=URL, params=PARAMS)
    DATA = R.json()
    
    try:
        Gartenlaube_Seiten = DATA["query"]["pages"]
        #print(DATA)

        for Pages in Gartenlaube_Seiten:
            print(Gartenlaube_Seiten[Pages])
            pagedet = {}
            pagedet["pageid"] = Gartenlaube_Seiten[Pages]["pageid"]
            pagedet["title"] = Gartenlaube_Seiten[Pages]["title"]
            pagedet["proofread"] = Gartenlaube_Seiten[Pages]["proofread"]
            pagedet["lastrevid"] = Gartenlaube_Seiten[Pages]["lastrevid"]
            catPages.append(pagedet)

        try:
            PARAMS["gcmcontinue"] = DATA["continue"]["gcmcontinue"]
            R = S.get(url=URL, params=PARAMS)
            DATA = R.json()
            Gartenlaube_Seiten = DATA["query"]["pages"]
            for Pages in Gartenlaube_Seiten:
                pagedet = {}
                pagedet["pageid"] = Gartenlaube_Seiten[Pages]["pageid"]
                pagedet["title"] = Gartenlaube_Seiten[Pages]["title"]
                pagedet["proofread"] = Gartenlaube_Seiten[Pages]["proofread"]
                pagedet["lastrevid"] = Gartenlaube_Seiten[Pages]["lastrevid"]
                catPages.append(pagedet)
                #print(Pages["title"])
        except KeyError:
            pass

        
    except KeyError:
        pass
    
    retDict[SubCat["title"]]=catPages
pages.append(retDict)
print(pages)


{'pageid': 180682, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1853) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:45:41Z', 'lastrevid': 2964676, 'length': 3244}
{'pageid': 197740, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1853) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:46:41Z', 'lastrevid': 3272772, 'length': 4701}
{'pageid': 197741, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1853) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:46:41Z', 'lastrevid': 3272773, 'length': 6795}
{'pageid': 197742, 'ns': 102, 'title': 'Sei

{'pageid': 342712, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1854) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:21Z', 'lastrevid': 3631782, 'length': 3078}
{'pageid': 348254, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1854) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:26Z', 'lastrevid': 3273322, 'length': 8924}
{'pageid': 348256, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1854) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:26Z', 'lastrevid': 3273323, 'length': 8454}
{'pageid': 348073, 'ns': 102, 'title': 'Sei

{'pageid': 154239, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1855) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:27Z', 'lastrevid': 3534090, 'length': 5516}
{'pageid': 154240, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1855) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:27Z', 'lastrevid': 3534091, 'length': 8773}
{'pageid': 154241, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1855) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:27Z', 'lastrevid': 3534092, 'length': 8756}
{'pageid': 154242, 'ns': 102, 'title': 'Sei

{'pageid': 316126, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1856) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:49:49Z', 'lastrevid': 3744212, 'length': 5493}
{'pageid': 314328, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1856) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:49:47Z', 'lastrevid': 3128543, 'length': 8718}
{'pageid': 314330, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1856) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:49:47Z', 'lastrevid': 3744182, 'length': 8186}
{'pageid': 314331, 'ns': 102, 'title': 'Sei

{'pageid': 351541, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1857) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:32Z', 'lastrevid': 3732236, 'length': 5752}
{'pageid': 351174, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1857) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:31Z', 'lastrevid': 3422636, 'length': 8296}
{'pageid': 351098, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1857) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:30Z', 'lastrevid': 3535994, 'length': 8115}
{'pageid': 397109, 'ns': 102, 'title': 'Sei

{'pageid': 158026, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1858) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:33Z', 'lastrevid': 3088080, 'length': 5941}
{'pageid': 158027, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1858) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:33Z', 'lastrevid': 3117791, 'length': 8657}
{'pageid': 158028, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1858) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:33Z', 'lastrevid': 3543547, 'length': 9285}
{'pageid': 158029, 'ns': 102, 'title': 'Sei

... (38 outputs left out)

{'pageid': 142305, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1897) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:10Z', 'lastrevid': 3295981, 'length': 3776}
{'pageid': 142306, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1897) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:10Z', 'lastrevid': 2954895, 'length': 9291}
{'pageid': 142307, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1897) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:44:10Z', 'lastrevid': 2954896, 'length': 8925}
{'pageid': 142308, 'ns': 102, 'title': 'Sei

{'pageid': 356807, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1898) 0000 b.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:43Z', 'lastrevid': 3628556, 'length': 11467}
{'pageid': 296416, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1898) 0000 c.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:49:27Z', 'lastrevid': 3296076, 'length': 320}
{'pageid': 296417, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1898) 0000 d.jpg', 'proofread': {'quality': 0, 'quality_text': 'Sofort fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:49:27Z', 'lastrevid': 3296077, 'length': 220}
{'pageid': 296869, 'ns': 102

{'pageid': 348555, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1899) 0000 a.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:27Z', 'lastrevid': 3665036, 'length': 469}
{'pageid': 357480, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1899) 0000 b.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:45Z', 'lastrevid': 3501154, 'length': 10536}
{'pageid': 348521, 'ns': 102, 'title': 'Seite:Die Gartenlaube (1899) 0001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'contentmodel': 'proofread-page', 'pagelanguage': 'de', 'pagelanguagehtmlcode': 'de', 'pagelanguagedir': 'ltr', 'touched': '2021-05-05T18:50:26Z', 'lastrevid': 3493218, 'length': 1074}
{'pageid': 357545, 'ns': 102, 'title

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



### 3. Für jede Seite die zugehörigen Text auslesen

* Für jeden Jahrgang wird ein JSON-File erzeugt nach dem Muster `GartenlaubeSeitenText_{Jahrgangskategorie}_{Timestamp}.json`und beinhaltet ein JSON-Objekt mit folgender Struktur:
```json
 [{"pageid" : {PAGEID},
   "title"   : {PAGETITLE},
   "html"    : {HTML_OUTPUT},
   "wikitext": {WIKI_MARKUP},
   "plaintxt": {mwparserfromhell(WIKI_MARKUP).strip_code)}
   }]
 ```

In [3]:
import mwparserfromhell
import time
import json

#print(pages[0]["Kategorie:Die Gartenlaube (1853)"])
#https://de.wikisource.org/w/api.php?action=parse&format=json&pageid=197745&prop=text%7Cwikitext

for page in pages[0]:
    SeitenText = []
    for wikisourcePage in pages[0][page]:
        print(wikisourcePage)
    
        PARAMS = {
        "action": "parse",
        "pageid": wikisourcePage["pageid"],
        "prop":"text|wikitext",
        "format": "json"
        }
        R = S.get(url=URL, params=PARAMS)
        DATA = R.json()
        SeitenTextDet = {}
        SeitenTextDet["pageid"] = wikisourcePage["pageid"]
        SeitenTextDet["title"] = wikisourcePage["title"]
        SeitenTextDet["proofread"] = wikisourcePage["proofread"]
        SeitenTextDet["lastrevid"] = wikisourcePage["lastrevid"]
        SeitenTextDet["html"] = DATA["parse"]["text"]["*"]
        SeitenTextDet["wikitext"] = DATA["parse"]["wikitext"]["*"]
        wikicode = mwparserfromhell.parse(SeitenTextDet["wikitext"] )
        SeitenTextDet["plaintxt"] = wikicode.strip_code()
        SeitenText.append(SeitenTextDet)

    f = open("output/GartenlaubeSeitenText_"+page+"_"+str(time.time())[0:10]+".json", "w")
    print(json.dumps(SeitenText),file=f)
    f.close()      


{'pageid': 180682, 'title': 'Seite:Die Gartenlaube (1853) 001.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 2964676}
{'pageid': 197740, 'title': 'Seite:Die Gartenlaube (1853) 002.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272772}
{'pageid': 197741, 'title': 'Seite:Die Gartenlaube (1853) 003.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272773}
{'pageid': 197742, 'title': 'Seite:Die Gartenlaube (1853) 004.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272774}
{'pageid': 197743, 'title': 'Seite:Die Gartenlaube (1853) 005.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3470094}
{'pageid': 197744, 'title': 'Seite:Die Gartenlaube (1853) 006.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272775}
{'pageid': 197745, 'title': 'Seite:Die Gartenlaube (1853) 007.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastre

{'pageid': 197797, 'title': 'Seite:Die Gartenlaube (1853) 058.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272826}
{'pageid': 197798, 'title': 'Seite:Die Gartenlaube (1853) 059.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272827}
{'pageid': 197799, 'title': 'Seite:Die Gartenlaube (1853) 060.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272828}
{'pageid': 197800, 'title': 'Seite:Die Gartenlaube (1853) 061.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272829}
{'pageid': 197801, 'title': 'Seite:Die Gartenlaube (1853) 062.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272830}
{'pageid': 197802, 'title': 'Seite:Die Gartenlaube (1853) 063.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272831}
{'pageid': 197803, 'title': 'Seite:Die Gartenlaube (1853) 064.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastre

{'pageid': 197854, 'title': 'Seite:Die Gartenlaube (1853) 115.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272878}
{'pageid': 197855, 'title': 'Seite:Die Gartenlaube (1853) 116.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3634068}
{'pageid': 197856, 'title': 'Seite:Die Gartenlaube (1853) 117.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272880}
{'pageid': 197857, 'title': 'Seite:Die Gartenlaube (1853) 118.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272881}
{'pageid': 197858, 'title': 'Seite:Die Gartenlaube (1853) 119.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3634072}
{'pageid': 197859, 'title': 'Seite:Die Gartenlaube (1853) 120.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3662357}
{'pageid': 197860, 'title': 'Seite:Die Gartenlaube (1853) 121.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastre

{'pageid': 197911, 'title': 'Seite:Die Gartenlaube (1853) 172.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272932}
{'pageid': 197912, 'title': 'Seite:Die Gartenlaube (1853) 173.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632278}
{'pageid': 197913, 'title': 'Seite:Die Gartenlaube (1853) 174.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272934}
{'pageid': 197914, 'title': 'Seite:Die Gartenlaube (1853) 175.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632279}
{'pageid': 197915, 'title': 'Seite:Die Gartenlaube (1853) 176.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632149}
{'pageid': 197916, 'title': 'Seite:Die Gartenlaube (1853) 177.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3727788}
{'pageid': 197917, 'title': 'Seite:Die Gartenlaube (1853) 178.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastre

{'pageid': 197968, 'title': 'Seite:Die Gartenlaube (1853) 229.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632446}
{'pageid': 197969, 'title': 'Seite:Die Gartenlaube (1853) 230.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3581149}
{'pageid': 197970, 'title': 'Seite:Die Gartenlaube (1853) 231.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632447}
{'pageid': 197971, 'title': 'Seite:Die Gartenlaube (1853) 232.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632451}
{'pageid': 197972, 'title': 'Seite:Die Gartenlaube (1853) 233.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632453}
{'pageid': 197973, 'title': 'Seite:Die Gartenlaube (1853) 234.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3272994}
{'pageid': 197974, 'title': 'Seite:Die Gartenlaube (1853) 235.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastre

{'pageid': 198024, 'title': 'Seite:Die Gartenlaube (1853) 286.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632627}
{'pageid': 198025, 'title': 'Seite:Die Gartenlaube (1853) 287.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3273043}
{'pageid': 198026, 'title': 'Seite:Die Gartenlaube (1853) 288.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632628}
{'pageid': 198027, 'title': 'Seite:Die Gartenlaube (1853) 289.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632630}
{'pageid': 198028, 'title': 'Seite:Die Gartenlaube (1853) 290.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3273046}
{'pageid': 198029, 'title': 'Seite:Die Gartenlaube (1853) 291.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3632633}
{'pageid': 198030, 'title': 'Seite:Die Gartenlaube (1853) 292.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastre

... (705 outputs left out)

{'pageid': 358106, 'title': 'Seite:Die Gartenlaube (1899) 0720.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3491749}
{'pageid': 358539, 'title': 'Seite:Die Gartenlaube (1899) 0721.jpg', 'proofread': {'quality': 3, 'quality_text': 'Korrigiert'}, 'lastrevid': 3297619}
{'pageid': 357478, 'title': 'Seite:Die Gartenlaube (1899) 0722.jpg', 'proofread': {'quality': 3, 'quality_text': 'Korrigiert'}, 'lastrevid': 3297620}
{'pageid': 358081, 'title': 'Seite:Die Gartenlaube (1899) 0723.jpg', 'proofread': {'quality': 3, 'quality_text': 'Korrigiert'}, 'lastrevid': 3762214}
{'pageid': 358608, 'title': 'Seite:Die Gartenlaube (1899) 0724.jpg', 'proofread': {'quality': 3, 'quality_text': 'Korrigiert'}, 'lastrevid': 3762202}
{'pageid': 358190, 'title': 'Seite:Die Gartenlaube (1899) 0725.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297623}
{'pageid': 357971, 'title': 'Seite:Die Gartenlaube (1899) 0726.jpg', 'proofread': {'quality': 1,

{'pageid': 358466, 'title': 'Seite:Die Gartenlaube (1899) 0770.jpg', 'proofread': {'quality': 3, 'quality_text': 'Korrigiert'}, 'lastrevid': 3774427}
{'pageid': 357524, 'title': 'Seite:Die Gartenlaube (1899) 0771.jpg', 'proofread': {'quality': 3, 'quality_text': 'Korrigiert'}, 'lastrevid': 3774909}
{'pageid': 358181, 'title': 'Seite:Die Gartenlaube (1899) 0772 a.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297675}
{'pageid': 358390, 'title': 'Seite:Die Gartenlaube (1899) 0772 b.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297676}
{'pageid': 358346, 'title': 'Seite:Die Gartenlaube (1899) 0772 c.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3665073}
{'pageid': 358005, 'title': 'Seite:Die Gartenlaube (1899) 0772 d.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297678}
{'pageid': 346572, 'title': 'Seite:Die Gartenlaube (1899) 0772 e.jpg', 'proofread': {'qual

{'pageid': 358055, 'title': 'Seite:Die Gartenlaube (1899) 0815.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297722}
{'pageid': 358500, 'title': 'Seite:Die Gartenlaube (1899) 0816.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297723}
{'pageid': 358219, 'title': 'Seite:Die Gartenlaube (1899) 0817.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297724}
{'pageid': 358183, 'title': 'Seite:Die Gartenlaube (1899) 0818.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297725}
{'pageid': 358233, 'title': 'Seite:Die Gartenlaube (1899) 0819.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297726}
{'pageid': 358091, 'title': 'Seite:Die Gartenlaube (1899) 0820.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297727}
{'pageid': 346578, 'title': 'Seite:Die Gartenlaube (1899) 0821.jpg', 'proofread': {'qual

{'pageid': 346580, 'title': 'Seite:Die Gartenlaube (1899) 0853.jpg', 'proofread': {'quality': 4, 'quality_text': 'Fertig'}, 'lastrevid': 3495157}
{'pageid': 357968, 'title': 'Seite:Die Gartenlaube (1899) 0854.jpg', 'proofread': {'quality': 3, 'quality_text': 'Korrigiert'}, 'lastrevid': 3094935}
{'pageid': 358594, 'title': 'Seite:Die Gartenlaube (1899) 0855.jpg', 'proofread': {'quality': 1, 'quality_text': 'Unkorrigiert'}, 'lastrevid': 3297768}


### 4. Merge der Jahrgangs-Files

* Die bestehenden Jahrgangs-Jsonfiles werden hier zu einer Gesamtdatei gemerged. 
* **Achtung**: Dieser Schritt läuft nur als python-Skript via Shell, nicht im Notebook.

In [4]:
import glob
import time

read_files = glob.glob("/home/paws/Gartenlaube/output/*.json")
print(read_files)

with open("/home/paws/Gartenlaube/output/GartenlaubeSeiten_Text_"+str(time.time())[0:10]+".json", "wb") as outfile:
    outfile.write(b",".join([open(f, "rb").read() for f in read_files]))


[]


FileNotFoundError: [Errno 2] No such file or directory: '/home/paws/Gartenlaube/output/GartenlaubeSeiten_Text_1640662892.json'